# Leaf Detection Machine Learning Notebook


In [ ]:
import os
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib
from utils.preprocessing import load_image, image_to_feature_vector

BASE_DIR = os.path.abspath(os.path.dirname("__file__"))
PROJECT_ROOT = BASE_DIR
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset")
MODEL_DIR = os.path.join(PROJECT_ROOT, "model")
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
def load_dataset(dataset_dir):
    X, y = [], []
    if not os.path.exists(dataset_dir):
        print(f"Dataset folder not found: {dataset_dir}")
        return np.array(X), np.array(y)

    classes = sorted([d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))])
    if not classes:
        print("No class folders found.")
        return np.array(X), np.array(y)

    for label in classes:
        class_dir = os.path.join(dataset_dir, label)
        for fname in os.listdir(class_dir):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                path = os.path.join(class_dir, fname)
                try:
                    img = load_image(path, size=(128, 128))
                    feat = image_to_feature_vector(img, resize_flatten=True, bins=16)
                    X.append(feat)
                    y.append(label)
                except Exception as e:
                    print("Skipping", path, ":", e)

    return np.array(X), np.array(y)

In [ ]:
print("Loading dataset...")
X, y = load_dataset(DATASET_DIR)

if len(X) == 0:
    print("No data found.")
else:
    print("Samples:", X.shape, "Labels:", len(y))

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

In [ ]:
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
print("Training model...")
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
model_path = os.path.join(MODEL_DIR, "leaf_model.joblib")
le_path = os.path.join(MODEL_DIR, "label_encoder.joblib")
joblib.dump(clf, model_path)
joblib.dump(le, le_path)
print("Saved model ->", model_path)
print("Saved label encoder ->", le_path)